# Homework: Build and Explain a Stock Forecasting System
**Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026**

[Read the full assignment](https://mles-course-vgqg4abvka-de.a.run.app/notes/homework/stock-market-prediction.html)

## Start here
1. Save a copy to your Drive (Colab) or download this notebook (Jupyter).
2. Select a CPU runtime. No paid API, trading account, or GPU is required.
3. Read the Markdown before running each cell, in order.
4. Choose your stock/ETF and document its data source. This is **not** a trading recommendation.
5. Study the worked scaffolding, complete the student explanations, and justify your decisions. Running unchanged code is not a complete submission.
6. Keep the final test gate closed until you have selected your model using validation data.

**Student starter:** saved outputs are intentionally empty. Run the cells to obtain your own results; there are no precomputed performance claims. Download availability can vary. Preserve your actual environment and dataset; use the documented CSV-upload route if online retrieval fails. Downloads use Python's standard HTTPS client rather than a browser-impersonation package. Start with a fresh runtime.

**Deliverables:** executed notebook, reproducibility files, original-unit predictions and metrics, interpretation, and a **five-minute English video**. Due date: instructor announcement.


## 1. Prediction contract and student decisions
At the close of trading observation t, predict the close at the next trading observation t+1 using only information available by t. With L observations, the input is [P(t-L+1), ..., P(t)] and the target is P(t+1).

**Task A:** Before running, write what information is available, why a random split is inappropriate, and what a persistence model predicts. Compare lookbacks 10 and 20 only on validation data. Test results are not tuning feedback.

The sample ticker below is a demonstration, not a recommended investment. Change it if you prefer. The fixed end date excludes incomplete current-day prices. For uploaded data, supply provenance and the adjustment convention yourself.


In [ ]:
import os, sys, subprocess, importlib.util
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # CPU-first teaching example; set before TensorFlow import.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
packages = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn', 'tensorflow':'tensorflow', 'joblib':'joblib'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import json, hashlib, random, platform, shutil
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import urllib.request, urllib.parse
DOWNLOAD_BACKEND = 'Python standard-library HTTPS (urllib); normal certificate verification'
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
WORK = Path('stock_hw_work')
DATA = WORK / 'data'
FIGURES = WORK / 'figures'
for folder in (WORK, DATA, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)
environment = {'python': platform.python_version(), **{p: version(p) for p in packages.values()}}
(WORK / 'environment.json').write_text(json.dumps(environment, indent=2))
(WORK / 'requirements.txt').write_text('\n'.join(f'{p}=={version(p)}' for p in packages.values()) + '\n')
print(environment)
print('A seed documents the experiment; exact results can still vary by platform/runtime.')


In [ ]:
# STUDENT: choose one instrument and explain your choice in the next Markdown cell.
TICKER = 'SPY'
START = '2021-01-01'
END = '2026-10-17'  # Exclusive; no unfinished current trading session.
MODE = 'download'  # 'download' or 'upload'
CURRENCY = 'USD'  # Change for other markets; this is a student declaration.
UPLOAD_PATH = ''  # Local Jupyter: path to your CSV. Colab: leave blank to show upload dialog.
UPLOAD_SOURCE_URL = ''  # Required in upload mode.
UPLOAD_PRICE_CONVENTION = ''  # Example: vendor-adjusted close; explain exactly what it means.
LOOKBACKS = (10, 20)
HIDDEN_UNITS = 16
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 0.001
RIDGE_ALPHAS = (0.1, 1.0, 10.0)
TRAIN_RATIO, VALID_RATIO = 0.70, 0.15
ANSWERS = {
    'prediction_contract_and_ticker': '',
    'leakage_prevention': '',
    'window_alignment': '',
    'validation_choice': '',
    'why_lstm_won_or_lost': '',
    'largest_error_analysis': '',
    'interpretability_and_limitations': '',
    'ai_assistance_and_personal_checks': '',
    'video_url_or_filename': ''
}
print('Fill ANSWERS with your own explanations before exporting your final submission.')


### Your prediction contract
**Replace this paragraph:** identify the instrument, market/currency, forecast horizon, available features, and why you chose this dataset. State your expectations **before** seeing results. You may also put this explanation into ANSWERS above.


## 2. Acquire a frozen dataset, with an upload fallback
**Download route:** fetch daily adjusted closing prices from Yahoo's chart endpoint through Python's standard HTTPS client. This is an unofficial data interface, not an official exchange feed, and may be unavailable or rate-limited. Do not repeatedly retry a failing service.

**Upload route:** set MODE to 'upload', fill the source URL/price convention/currency, then upload a CSV with columns **date, close** (case-insensitive). Obtain at least **750 trading observations** from a source you may use. Never substitute synthetic prices for the final experiment.

The original provider file and a canonical two-column snapshot are saved separately. Later runs reuse the snapshot rather than silently changing the data. To study a different acquisition, use a separate working folder and preserve both experiments. Do not assume that retrospective adjustments are point-in-time historical information. Respect redistribution restrictions when sharing files.

**Compatibility:** downloading uses the Python standard library with normal HTTPS certificate verification. It does not require yfinance, curl-cffi, browser impersonation, an API key, or changes to browser security settings. If acquisition fails, use the CSV route instead of repeatedly retrying.


In [ ]:
snapshot = DATA / 'prices.csv'
provenance_path = DATA / 'provenance.json'
request_identity = {'ticker': TICKER, 'start': START, 'end_exclusive': END, 'mode': MODE,
                    'currency': CURRENCY, 'upload_source_url': UPLOAD_SOURCE_URL,
                    'upload_price_convention': UPLOAD_PRICE_CONVENTION}
if snapshot.exists():
    if not provenance_path.exists():
        raise ValueError('Snapshot lacks provenance. Restore its metadata rather than guessing its source.')
    provenance = json.loads(provenance_path.read_text())
    if provenance['request'] != request_identity:
        raise ValueError('Configuration differs from the saved acquisition. Use a separate WORK folder.')
    df = pd.read_csv(snapshot)
    print('Reusing saved snapshot:', snapshot)
else:
    if MODE == 'download':
        endpoint = 'https://query1.finance.yahoo.com/v8/finance/chart/' + urllib.parse.quote(TICKER, safe='')
        params = {'period1':int(datetime.fromisoformat(START).replace(tzinfo=timezone.utc).timestamp()),
                  'period2':int(datetime.fromisoformat(END).replace(tzinfo=timezone.utc).timestamp()),
                  'interval':'1d', 'includeAdjustedClose':'true'}
        request_url = endpoint + '?' + urllib.parse.urlencode(params)
        request = urllib.request.Request(request_url, headers={'User-Agent':'MLES-course-educational-check/1.0'})
        try:
            with urllib.request.urlopen(request, timeout=20) as response:
                raw_bytes = response.read()
            payload = json.loads(raw_bytes)
            results = (payload.get('chart') or {}).get('result')
            if not results:
                raise ValueError('No chart result returned by the provider.')
            record = results[0]
            timestamps = record.get('timestamp') or []
            adjusted = record.get('indicators', {}).get('adjclose', [])
            if not timestamps or not adjusted or len(adjusted[0].get('adjclose', [])) != len(timestamps):
                raise ValueError('Adjusted closing prices are missing or misaligned; do not silently substitute raw prices.')
            provider_currency = record.get('meta', {}).get('currency')
            if provider_currency and provider_currency != CURRENCY:
                raise ValueError(f'Provider currency is {provider_currency}; update CURRENCY and use a separate work folder.')
            market_timezone = record.get('meta', {}).get('exchangeTimezoneName', 'UTC')
            dates = pd.to_datetime(timestamps, unit='s', utc=True).tz_convert(market_timezone).date
            df = pd.DataFrame({'date':dates, 'close':adjusted[0]['adjclose']})
        except Exception as exc:
            raise RuntimeError('Market data could not be acquired. Inspect the cause below, or use MODE=upload with a permitted CSV.') from exc
        original_path = DATA / 'provider_download.json'
        original_path.write_bytes(raw_bytes)
        source_url = 'https://finance.yahoo.com/quote/' + TICKER + '/history/'
        price_convention = 'Yahoo chart indicators.adjclose: vendor-adjusted closing price'
    elif MODE == 'upload':
        if not UPLOAD_SOURCE_URL or not UPLOAD_PRICE_CONVENTION:
            raise ValueError('Fill UPLOAD_SOURCE_URL and UPLOAD_PRICE_CONVENTION first.')
        original_path = DATA / 'uploaded_original.csv'
        if UPLOAD_PATH:
            original_path.write_bytes(Path(UPLOAD_PATH).read_bytes())
        else:
            try:
                from google.colab import files
            except ImportError as exc:
                raise RuntimeError('Local Jupyter: set UPLOAD_PATH to your CSV file.') from exc
            uploaded = files.upload()
            if len(uploaded) != 1:
                raise ValueError('Upload exactly one CSV.')
            original_path.write_bytes(next(iter(uploaded.values())))
        df = pd.read_csv(original_path)
        df.columns = [str(c).strip().lower() for c in df.columns]
        if not {'date', 'close'}.issubset(df.columns):
            raise ValueError('Your CSV needs date and close columns. Document any column conversion.')
        df = df[['date', 'close']].copy()
        source_url, price_convention = UPLOAD_SOURCE_URL, UPLOAD_PRICE_CONVENTION
    else:
        raise ValueError('MODE must be download or upload.')
    df.to_csv(snapshot, index=False)
    provenance = {'request': request_identity, 'source_url': source_url,
                  'retrieved_utc': datetime.now(timezone.utc).isoformat(),
                  'price_convention': price_convention,
                  'download_backend': DOWNLOAD_BACKEND if MODE == 'download' else 'user-provided CSV',
                  'original_file': original_path.name,
                  'original_sha256': hashlib.sha256(original_path.read_bytes()).hexdigest(),
                  'snapshot_sha256': hashlib.sha256(snapshot.read_bytes()).hexdigest()}
    provenance_path.write_text(json.dumps(provenance, indent=2))

actual_hash = hashlib.sha256(snapshot.read_bytes()).hexdigest()
if actual_hash != provenance['snapshot_sha256']:
    raise ValueError('The saved snapshot changed. Document a new version; do not silently reuse old provenance.')
print(json.dumps(provenance, indent=2))


## 3. Inspect data before modeling
This scaffold sorts dates but does **not** silently drop duplicates, fill missing prices, or invent observations. If a check stops you, inspect the original data, document your decision, and create a traceable corrected input. A source download and a cleaned table are different artifacts.

**Checkpoint:** explain missing trading dates, price units, adjustment conventions, and any corporate-action limitations. Do not interpolate using a future observation.


In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='raise').dt.tz_localize(None)
df['close'] = pd.to_numeric(df['close'], errors='raise')
if df[['date', 'close']].isna().any().any():
    raise ValueError('Missing date/close: inspect and document a cleaning decision.')
if df['date'].duplicated().any():
    raise ValueError('Duplicate trading dates: inspect and resolve explicitly.')
if not np.isfinite(df['close']).all() or (df['close'] <= 0).any():
    raise ValueError('Closing prices must be finite and positive.')
was_sorted = df['date'].is_monotonic_increasing
df = df.sort_values('date').reset_index(drop=True)
if len(df) < 750:
    raise ValueError(f'Found {len(df)} observations; at least 750 are required.')
df.to_csv(DATA / 'cleaned_prices.csv', index=False)
print('Already sorted:', was_sorted, '| rows:', len(df), '| currency:', CURRENCY)
print(df.head().to_string(index=False))
print(df.tail().to_string(index=False))


## 4. Split observations chronologically and fit preprocessing on training only
We assign examples using their **target row/date**. Training is the first 70% of observations, validation the next 15%, and test the last 15%. Rounding is shown explicitly.

Only training observations fit the scaler. Transforming a held-out value with a training-fitted scaler is permitted; letting a held-out value determine that scaler is not. The test data will not be used for choosing model settings.


In [ ]:
n = len(df)
train_end = int(n * TRAIN_RATIO)
valid_end = int(n * (TRAIN_RATIO + VALID_RATIO))
prices = df['close'].to_numpy(dtype=np.float64)
scaler = StandardScaler().fit(prices[:train_end].reshape(-1, 1))
scaled = scaler.transform(prices.reshape(-1, 1)).astype(np.float32)
split_record = []
for label, start, stop in [('train', 0, train_end), ('validation', train_end, valid_end), ('test', valid_end, n)]:
    split_record.append({'split':label, 'rows':stop-start,
                         'first_date':str(df.loc[start,'date'].date()),
                         'last_date':str(df.loc[stop-1,'date'].date())})
print(pd.DataFrame(split_record).to_string(index=False))
(WORK / 'split_dates.json').write_text(json.dumps(split_record, indent=2))
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(df['date'], prices, color='#142b3b', linewidth=1)
for label, start, stop, color in [('Train',0,train_end,'#b6d8cd'),
                                 ('Validation',train_end,valid_end,'#edd7a5'), ('Test',valid_end,n,'#d9c4d4')]:
    ax.axvspan(df.loc[start,'date'], df.loc[stop-1,'date'], alpha=.45, color=color, label=label)
ax.set(title=f'{TICKER}: chronological observation splits', ylabel=f'Closing price ({CURRENCY})')
ax.legend(); fig.tight_layout(); fig.savefig(FIGURES / '01_splits.png'); plt.show()


## 5. Construct windows and prove their alignment
For target row j, input rows are j-L through j-1. Row j must **never** appear in its own input. Both lookbacks use the same target-row set, starting at row 20, so the controlled comparison changes history length rather than evaluation dates.

A validation window may include earlier training observations. Later test windows may include already observed earlier test prices: that is rolling **one-step** forecasting, not a month-ahead recursive forecast. Model weights remain fixed during test evaluation.

**Task B:** Use the displayed example to explain the array shape (examples, time steps, features) and the difference between input date and target date.


In [ ]:
targets = np.arange(max(LOOKBACKS), n)
train_mask = targets < train_end
valid_mask = (targets >= train_end) & (targets < valid_end)
test_mask = targets >= valid_end
y_scaled = scaled[targets, 0]
y_original = prices[targets]
windows = {L: np.stack([scaled[j-L:j] for j in targets]) for L in LOOKBACKS}
for L, X in windows.items():
    print('L=', L, 'shape=', X.shape, 'train/validation/test=',
          (int(train_mask.sum()), int(valid_mask.sum()), int(test_mask.sum())))
example_j = int(targets[valid_mask][0])
example_dates = df.loc[example_j-20:example_j-1, ['date','close']]
print('Example: first validation target')
print(example_dates.to_string(index=False))
print('Target:', df.loc[example_j, ['date','close']].to_dict())
print('Input ends before target:', df.loc[example_j-1,'date'] < df.loc[example_j,'date'])

def original_units(values):
    return scaler.inverse_transform(np.asarray(values).reshape(-1,1)).ravel()

def scores(truth, prediction):
    residual = np.asarray(prediction).ravel() - np.asarray(truth).ravel()
    return {'MAE':float(np.mean(np.abs(residual))), 'RMSE':float(np.sqrt(np.mean(residual**2)))}

baseline_valid = prices[targets[valid_mask]-1]
validation_baseline = scores(y_original[valid_mask], baseline_valid)
print('Persistence validation:', validation_baseline)


## 6. Train and select a Ridge baseline using validation data
Ridge sees the same 20 past closing prices, flattened into 20 features. It does not see the next close. The target is scaled with the same training-fitted scaler and predictions are converted back to the original price unit before scoring.

The small regularization grid below is selected on validation MAE. Do not expand it after viewing test scores.


In [ ]:
X20 = windows[20].reshape(len(targets), 20)
ridge_candidates = {}
ridge_rows = []
for alpha in RIDGE_ALPHAS:
    candidate = Ridge(alpha=alpha).fit(X20[train_mask], y_scaled[train_mask])
    pred = original_units(candidate.predict(X20[valid_mask]))
    ridge_candidates[alpha] = candidate
    ridge_rows.append({'alpha':alpha, **scores(y_original[valid_mask], pred)})
ridge_validation = pd.DataFrame(ridge_rows).sort_values(['MAE','alpha'])
selected_alpha = float(ridge_validation.iloc[0]['alpha'])
ridge_model = ridge_candidates[selected_alpha]
ridge_validation.to_csv(WORK / 'ridge_validation.csv', index=False)
print(ridge_validation.to_string(index=False))
print('Selected Ridge alpha:', selected_alpha)


## 7. Train small LSTMs: lookbacks 10 versus 20
The model reads a sequence and returns one scaled price prediction. The LSTM state is reset between independent windows (stateful=False); we do not carry state across test examples. Early stopping watches **validation loss**, never test loss.

We reset the seed for each candidate, keep the architecture/optimizer/epoch budget fixed, and compare the same target dates. Changing the lookback changes the information window, not the number of features. The result is a sensitivity experiment, not a causal explanation.

**Task C:** Before training, predict which lookback will work better and explain why. Afterwards, compare that expectation with the measured validation MAE. A prediction that turns out wrong is acceptable if you explain it honestly.


In [ ]:
lstm_candidates, histories, lstm_rows = {}, {}, []
for L in LOOKBACKS:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(L,1)),
        tf.keras.layers.LSTM(HIDDEN_UNITS, stateful=False),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
    history = model.fit(windows[L][train_mask], y_scaled[train_mask],
                        validation_data=(windows[L][valid_mask], y_scaled[valid_mask]),
                        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False, verbose=0,
                        callbacks=[tf.keras.callbacks.EarlyStopping(
                            monitor='val_loss', patience=3, restore_best_weights=True)])
    pred = original_units(model.predict(windows[L][valid_mask], verbose=0))
    lstm_candidates[L], histories[L] = model, history.history
    lstm_rows.append({'lookback':L, 'epochs_run':len(history.history['loss']),
                      **scores(y_original[valid_mask], pred)})
    print('Finished lookback', L, lstm_rows[-1])

lstm_validation = pd.DataFrame(lstm_rows).sort_values(['MAE','lookback'])
selected_L = int(lstm_validation.iloc[0]['lookback'])
lstm_model = lstm_candidates[selected_L]
lstm_validation.to_csv(WORK / 'lstm_validation.csv', index=False)
print(lstm_validation.to_string(index=False))
print('Selected LSTM lookback:', selected_L)
lstm_model.summary()
fig, axes = plt.subplots(1,2, figsize=(12,4))
for ax, L in zip(axes, LOOKBACKS):
    h = histories[L]
    epoch_numbers = np.arange(1, len(h['loss'])+1)
    ax.plot(epoch_numbers, h['loss'], label='Training')
    ax.plot(epoch_numbers, h['val_loss'], label='Validation')
    ax.set(title=f'Lookback {L}', xlabel='Epoch', ylabel='MSE (standardized price units)')
    ax.legend()
fig.tight_layout(); fig.savefig(FIGURES / '02_learning_curves.png'); plt.show()


## 8. Freeze your decisions before opening the test gate
**Stop and write:** use only the validation evidence above to explain your selected lookback and settings. Fill ANSWERS['validation_choice']. Do not claim the longer window must be better.

The following cell records the selected configuration. It does not retrain using validation data. Training-only fitted models and scaler remain fixed. Inspect your method, not the held-out answers, before continuing.


In [ ]:
selection = {'ticker':TICKER, 'seed':SEED, 'selected_lookback':selected_L,
             'hidden_units':HIDDEN_UNITS, 'max_epochs':EPOCHS, 'batch_size':BATCH_SIZE,
             'learning_rate':LEARNING_RATE, 'loss':'mse', 'optimizer':'Adam',
             'early_stopping':{'monitor':'val_loss','patience':3,'restore_best_weights':True},
             'ridge_alpha':selected_alpha, 'selection_metric':'validation MAE in original price units',
             'lookbacks_compared':list(LOOKBACKS), 'ridge_alphas_compared':list(RIDGE_ALPHAS),
             'snapshot_sha256':actual_hash, 'split_dates':split_record}
(WORK / 'selection.json').write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))

FINALIZE_TEST = False  # STUDENT: change to True only after documenting and freezing decisions.
predictions = None
if not FINALIZE_TEST:
    print('Test gate CLOSED. Complete your validation explanation, then intentionally reopen this cell.')
else:
    if not ANSWERS['validation_choice'].strip():
        raise ValueError('Write your validation-based selection explanation in ANSWERS first.')
    test_targets = targets[test_mask]
    predictions = pd.DataFrame({
        'target_date':df.loc[test_targets,'date'].to_numpy(),
        'previous_close':prices[test_targets-1],
        'observed_close':prices[test_targets],
        'persistence_prediction':prices[test_targets-1],
        'ridge_prediction':original_units(ridge_model.predict(X20[test_mask])),
        'lstm_prediction':original_units(lstm_model.predict(windows[selected_L][test_mask], verbose=0))
    })
    metric_rows = [{'model':name, **scores(predictions['observed_close'], predictions[column])}
                   for name,column in [('Persistence','persistence_prediction'),
                                       ('Ridge','ridge_prediction'),('LSTM','lstm_prediction')]]
    metrics = pd.DataFrame(metric_rows)
    predictions.to_csv(WORK / 'test_predictions.csv', index=False)
    metrics.to_csv(WORK / 'test_metrics.csv', index=False)
    (WORK / 'test_evaluation.json').write_text(json.dumps({
        'evaluated_utc':datetime.now(timezone.utc).isoformat(),
        'note':'Do not tune settings after consulting these held-out results.',
        'selection':selection}, indent=2))
    print(metrics.to_string(index=False))
    print('All models use',len(predictions),'identical target dates; units:',CURRENCY)


## 9. Inspect aligned predictions and one failure case
After the test gate is intentionally opened, examine the full held-out interval and a zoomed view. Identify the largest absolute LSTM error, and compare it with persistence on the **same date**.

**Task D:** Is the LSTM better on MAE and RMSE? Does that justify a statement about trading profit or the direction of tomorrow's movement? Explain why not. Discuss underfitting/overfitting and the limits of a single instrument and a single test period. Do not invent news explanations for an error.


In [ ]:
if predictions is None:
    print('Complete the validation explanation and open the test gate before this section.')
else:
    fig, axes = plt.subplots(2,1,figsize=(12,8))
    for ax, view, title in [(axes[0],predictions,'All test targets'),
                            (axes[1],predictions.tail(60),'Last 60 test targets')]:
        ax.plot(view['target_date'],view['observed_close'],label='Observed',color='black',linewidth=2)
        for column,label in [('persistence_prediction','Persistence'),('ridge_prediction','Ridge'),
                             ('lstm_prediction','LSTM')]:
            ax.plot(view['target_date'],view[column],label=label,alpha=.8)
        ax.set(title=title,ylabel=f'Price ({CURRENCY})'); ax.legend()
    fig.tight_layout(); fig.savefig(FIGURES / '03_test_forecasts.png'); plt.show()
    worst_index = (predictions['lstm_prediction']-predictions['observed_close']).abs().idxmax()
    failure = predictions.loc[worst_index].copy()
    print('Largest absolute LSTM test error:')
    print(failure.to_string())
    print('Absolute LSTM error:',abs(failure['lstm_prediction']-failure['observed_close']))
    print('Absolute persistence error:',abs(failure['persistence_prediction']-failure['observed_close']))
    failure.to_csv(WORK / 'largest_error_case.csv', header=['value'])


## 10. Demonstrate the prediction system on an actual held-out window
The function below takes the row number of a test-period target and returns the available input dates, the three forecasts, and the later-observed target. It does **not** use the target price to create forecasts. Displaying the observed target afterwards is evaluation, not an input feature.

For your video, show an input window and describe the complete path from CSV to a prediction. This is a historical held-out demonstration, not live market execution.


In [ ]:
def forecast_for_test_target(target_row):
    if predictions is None:
        raise RuntimeError('The final test gate is still closed.')
    j = int(target_row)
    if not valid_end <= j < n:
        raise ValueError('Choose a target row inside the held-out test period.')
    x_lstm = scaled[j-selected_L:j].reshape(1,selected_L,1)
    x_ridge = scaled[j-20:j].reshape(1,20)
    forecast = {'target_date':str(df.loc[j,'date'].date()),
                'input_start':str(df.loc[j-selected_L,'date'].date()),
                'input_end':str(df.loc[j-1,'date'].date()),
                'persistence_prediction':float(prices[j-1]),
                'ridge_prediction':float(original_units(ridge_model.predict(x_ridge))[0]),
                'lstm_prediction':float(original_units(lstm_model.predict(x_lstm,verbose=0))[0])}
    print('Forecasts using past observations only:',json.dumps(forecast,indent=2))
    print('Observed target, revealed for evaluation:',float(prices[j]))
    return forecast

if predictions is not None:
    demo = forecast_for_test_target(valid_end)
else:
    print('The demonstration becomes available after you intentionally evaluate the final test set.')


## 11. Explain, disclose, and package your work
**Download fallback:** if a Colab output frame cannot load its JavaScript, do not weaken browser security. Open the left **Files** panel, locate **stock_forecasting_submission.zip**, right-click it, and choose **Download**. Individual PNG figures are in **stock_hw_work/figures** and can also be downloaded there. The default package stays DRAFT until you complete the required work; download your executed notebook separately through **File > Download > Download .ipynb**.

Fill all entries in ANSWERS with your own evidence-based explanations. Include the actual AI tool/model, representative prompts, what you changed, and what you personally checked. Write 'No AI assistance' if appropriate. Do not paste secrets into your notebook or disclosure.

Explain what the LSTM is intended to remember, and why the lookback experiment does not identify a causal driver of stock prices. No attention map or attribution chart is required.

**Important:** the export below packages your current files; it does not judge whether your explanations are correct. Empty answers or a closed test gate produce a DRAFT package. Include the separately downloaded **executed notebook** in your final submission. The archive can contain licensed market data: share it only through permitted course submission channels.


In [ ]:
# STUDENT: update ANSWERS in section 1, then run this export after recording your video.
missing_answers = [key for key,value in ANSWERS.items() if not str(value).strip()]
status = 'draft' if predictions is None or missing_answers else 'student-completed; not automatically validated'
(WORK / 'student_explanations.json').write_text(json.dumps(ANSWERS,indent=2))
(WORK / 'submission_status.json').write_text(json.dumps({
    'status':status, 'missing_answers':missing_answers,
    'test_evaluated_this_run':predictions is not None,
    'reminder':'Add the executed .ipynb and your video. Check data redistribution permissions.'},indent=2))
joblib.dump({'ridge':ridge_model, 'scaler':scaler, 'selection':selection}, WORK / 'ridge_and_preprocessing.joblib')
lstm_model.save(WORK / 'lstm.keras')
(WORK / 'README.txt').write_text(
    'Stock forecasting HW - Machine Learning in Engineering Science, Fall 2026\n'
    '1. Include your executed stock_forecasting.ipynb with this package.\n'
    '2. Install requirements.txt in an appropriate Python environment.\n'
    '3. Preserve the CSV and provenance; use the original working-folder layout.\n'
    '4. Run the notebook in order; inspect selection.json and split_dates.json.\n'
    '5. Model selection uses validation only; final test results are not tuning feedback.\n'
    '6. See student_explanations.json for interpretation, AI use, and video location.\n'
    '7. Check dataset sharing permissions before distributing this archive.\n'
    'Status: '+status+'\n')
archive = shutil.make_archive('stock_forecasting_submission','zip',WORK)
print('Package:',archive,'| Status:',status)
print('Missing answers:',missing_answers)
print('Download the executed notebook separately and add it to your final submission.')
# Colab: uncomment the following two lines when ready to download the archive.
# from google.colab import files
# files.download(archive)


## 12. Your five-minute English video
| Time | Show and explain |
|---|---|
| 0:00-0:30 | Instrument, forecast horizon, available information, and research question. |
| 0:30-1:20 | Data provenance, split boundaries, one input/target example, and leakage prevention. |
| 1:20-2:20 | Persistence, Ridge, LSTM input shape, architecture, and validation-based lookback choice. |
| 2:20-3:35 | A historical held-out system demonstration, score table, and aligned forecast plots. |
| 3:35-4:30 | Largest error, whether LSTM beat persistence, and what the result cannot establish. |
| 4:30-5:00 | Reproduction instructions, AI-use disclosure, and what you personally learned. |

Use your own narration. Aim for 5:00 (acceptable: 4:30-5:30). Show actual notebook results; do not spend the recording waiting for training. Webcam optional. Confirm your instructor can access the video.

### Final checklist
- [ ] All three models evaluated on identical held-out target dates, in original units.
- [ ] Training-only preprocessing; no test-based model selection.
- [ ] Input/target dates and array shapes explained.
- [ ] Lookback comparison, learning curves, aligned forecasts, and failure analysis included.
- [ ] Dataset fingerprint, provenance, environment, seed, and settings saved.
- [ ] All student explanations and the AI-use log completed honestly.
- [ ] Executed notebook, permitted data/reproduction route, exported results, and video submitted.

### Official technical references
- [TensorFlow time-series tutorial](https://www.tensorflow.org/tutorials/structured_data/time_series)
- [scikit-learn leakage pitfalls](https://scikit-learn.org/stable/common_pitfalls.html)
- [Ridge API](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)
- [Keras LSTM](https://keras.io/api/layers/recurrent_layers/lstm/)
- [Keras EarlyStopping](https://keras.io/api/callbacks/early_stopping/)
- [Python standard-library HTTPS requests](https://docs.python.org/3/library/urllib.request.html)

Educational forecasting only; no investment or profitability claim is supported merely by a price-prediction chart.
